In [1]:
!pip install pyvirtualdisplay
!pip install huggingface_sb3
!pip install stable_baselines3
!pip install swig
!pip install "gymnasium[box2d]"
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
transformers 5.13.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 99.6 MB/s eta 0:00:00


In [2]:
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400,900))
virtual_display.start()

In [3]:
import gymnasium

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import(
    notebook_login
)

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Example Code to test out the enviornment

Here we create an enviornment `(LunarLander-v3)`, do some random action at first and observe what happens to the env after doing that specific action.

In [4]:
# Import necessary library for this cell
import gymnasium as gym

# Create the env that the RL agent will be working with
env = gym.make('LunarLander-v3')

# Reset env
observation, info = env.reset()

for _ in range(20):
  # take a random action
  action = env.action_space.sample()
  print('Action taken:', action)

  # Do the action in the env and get the next state, reward, terminated, and info
  observation, reward, terminated, truncated, info = env.step(action)

  # If the game is terminated or truncated:
  if terminated or truncated:
    # reset the env
    print('Env is reset')
    observation, info = env.reset()

env.close()

Action taken: 3
Action taken: 1
Action taken: 2
Action taken: 1
Action taken: 2
Action taken: 1
Action taken: 0
Action taken: 3
Action taken: 2
Action taken: 0
Action taken: 0
Action taken: 2
Action taken: 0
Action taken: 0
Action taken: 2
Action taken: 0
Action taken: 0
Action taken: 2
Action taken: 1
Action taken: 1


<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


## Training our agent

### Observing our enviornment

In [5]:
env = gym.make('LunarLander-v3')
env.reset()
print('###### OBSERVATION SPACE ######\n')
print('Observation Space Shape:', env.observation_space.shape)
print('Sample Observation:', env.observation_space.sample()) # Get a random observation

###### OBSERVATION SPACE ######

Observation Space Shape: (8,)
Sample Observation: [-2.2326634  -0.944572    0.91314405  9.495315    5.720602    7.544739
  0.47939834  0.20178436]


Here we see that the space is `(8,)` whihc means that out observation is a vector size of 8:

- Horizontal Pad coordinate
- Vertical Pad Coordinate
- Horizontal Speed
- Vertical Space
- Angle
- Angular Speed
- 2 Booleans of if the left and right leg touched the ground.

In [6]:
print('\n ##### ACTION SPACE ##### \n')
# Find out the action space shape(How many actions are there?)
print(f'Action Space Shape:{env.action_space.n}')
# Print out a random action
print(f'Action Space Sample: {env.action_space.sample()}')


 ##### ACTION SPACE ##### 

Action Space Shape:4
Action Space Sample: 3


**Action list:**

- Action 0: Do nothing,
- Action 1: Fire left orientation engine,
- Action 2: Fire the main engine,
- Action 3: Fire right orientation engine.

**Reward Function**:
- Is increased/decreased the closer/further the lander is to the landing pad.
- Is increased/decreased the slower/faster the lander is moving.
- Is decreased the more the lander is tilted (angle not horizontal).
- Is increased by 10 points for each leg that is in contact with the ground.
- Is decreased by 0.03 points each frame a side engine is firing.
- Is decreased by 0.3 points each frame the main engine is firing.

The episode receive an additional reward of -100 or +100 points for crashing or landing safely respectively.


In [7]:
# We will create a vectorized env so that the agent can have a diverse experience during training

env = make_vec_env('LunarLander-v3')

### **Create the model!**

We will finally make the model to successfully land the lunar lander on the pad by controlling left, right, and main orientation engine.

We will first train a toy model to test out the MlpPolicy and PPO about 200000 times then we will get into training out actual agent 1000000 times!

In [ ]:
# Create our agent
model = PPO('MlpPolicy', env, verbose=1)

# Train the agent
model.learn(total_timesteps=int(2e5))

Using cuda device


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 90.5     |
|    ep_rew_mean     | -214     |
| time/              |          |
|    fps             | 505      |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 91.1         |
|    ep_rew_mean          | -198         |
| time/                   |              |
|    fps                  | 416          |
|    iterations           | 2            |
|    time_elapsed         | 9            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0044416217 |
|    clip_fraction        | 0.012        |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.38        |
|    explained_variance   | 0.00346      |
|    learning_r

In [8]:
# We will now define the PPO MlpPolicy acritecture
## We will use MultiLayerPerception policy(Mlp) because the input is a vector
## If we had frames as input we would use CnnPolicy as we need the capapbilities of a CNN

model = PPO(
    policy='MlpPolicy',
    env=env,
    batch_size=64,
    n_epochs=4,
    gamma=0.999, # Gamma tells the agent to either think short-term or long-term
    gae_lambda=0.98,
    ent_coef=0.01,
    verbose=1
)

Using cuda device


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Training the PPO agent

In [9]:
model.learn(total_timesteps=1000000)
model_name='ppo-LunarLander-v3'
model.save(model_name)

Streaming output truncated to the last 5000 lines.
|    value_loss           | 31.7        |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 439         |
|    ep_rew_mean          | 219         |
| time/                   |             |
|    fps                  | 412         |
|    iterations           | 252         |
|    time_elapsed         | 1251        |
|    total_timesteps      | 516096      |
| train/                  |             |
|    approx_kl            | 0.005187293 |
|    clip_fraction        | 0.0287      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.842      |
|    explained_variance   | 0.406       |
|    learning_rate        | 0.0003      |
|    loss                 | 1e+03       |
|    n_updates            | 1004        |
|    policy_gradient_loss | -0.00338    |
|    value_loss           | 1.16e+03    |
-------------------------

### Evaluate the model

- Env in a Monitor
- Check the performance
- Evaluate Policy

In [11]:
eval_env = Monitor(gym.make('LunarLander-v3'))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f'mean_reward={mean_reward:.2f} +/- {std_reward:.2f}')

mean_reward=246.07 +/- 65.54


In [14]:
notebook_login()
!git config --global credential.helper store

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [17]:
from re import M
import gymnasium as gym

from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.env_util import make_vec_env

from huggingface_sb3 import package_to_hub
env_id = 'LunarLander-v3'

model_architecture ='PPO'

# Change 'official_ak' to your Hugging Face username
repo_id = 'official-ak/ppo-LunarLander-v3' # TODO: Replace 'your_username' with your actual Hugging Face username

commit_message = 'Upload PPo LunarLander-v3 trained agent'

eval_env = DummyVecEnv([lambda: Monitor(gym.make(env_id, render_mode="rgb_array"))])

package_to_hub(
    model=model,
    model_name=model_name,
    model_architecture=model_architecture,
    env_id=env_id,
    eval_env=eval_env,
    repo_id=repo_id,
    commit_message=commit_message,
)

ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Saving video to /tmp/tmpmvb14lp1/-step-0-to-step-1000.mp4
Moviepy - Building video /tmp/tmpmvb14lp1/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmpmvb14lp1/-step-0-to-step-1000.mp4



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Moviepy - Done !
Moviepy - video ready /tmp/tmpmvb14lp1/-step-0-to-step-1000.mp4
✘ 'DummyVecEnv' object has no attribute 'video_recorder'
✘ We are unable to generate a replay of your agent, the package_to_hub
process continues
✘ Please open an issue at
https://github.com/huggingface/huggingface_sb3/issues
ℹ Pushing repo official-ak/ppo-LunarLander-v3 to the Hugging Face
Hub


...

  ...r-v3/policy.optimizer.pth: 100%|##########| 88.7kB / 88.7kB            

  ...LunarLander-v3/policy.pth: 100%|##########| 44.1kB / 44.1kB            

  ...e7/ppo-LunarLander-v3.zip: 100%|##########|  149kB /  149kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/official-ak/ppo-LunarLander-v3/tree/main/


CommitInfo(commit_url='https://huggingface.co/official-ak/ppo-LunarLander-v3/commit/2da371b7b283e3fb782251de22a585e239948537', commit_message='Upload PPo LunarLander-v3 trained agent', commit_description='', oid='2da371b7b283e3fb782251de22a585e239948537', pr_url=None, repo_url=RepoUrl('https://huggingface.co/official-ak/ppo-LunarLander-v3', endpoint='https://huggingface.co', repo_type='model', repo_id='official-ak/ppo-LunarLander-v3'), pr_revision=None, pr_num=None)